# Exploratory CodeT5 experiment

This separate experiment asks whether a pretrained code encoder can learn our five author labels. It is **not part of the main four-model ablation comparison** and does not reproduce the paper's CodeT5-Authorship model.

We fine-tune `Salesforce/codet5-base` with masked mean pooling and a linear five-class head, using original code with comments. **We did not run a comment-removal ablation for CodeT5.** The 500-feature experiment applies only to TF-IDF classifiers.

Use a fresh Google Colab A100 GPU runtime. Run section 1, restart the session once, then run sections 2-10 in order. Training is enabled in section 2; review the timing estimate in section 7 before running section 8.

The published notebook is a complete rerunnable source notebook with cleared outputs, not an executed-notebook export. Verified outputs from the completed run are in this folder's `results/` directory. AI assisted in preparing the notebook.


## 1. Install only the model library
Use a fresh Colab runtime, especially after earlier NumPy installation errors. This cell does not install the project's requirements or upgrade NumPy, pandas, scikit-learn or PyTorch. After installation, restart the Colab session once and continue from section 2. This ensures Python uses the pinned version rather than a previously imported version.

In [ ]:
%pip -q install transformers==4.57.1


## 2. Imports, GPU check and settings

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import random
import shutil
import time
import urllib.request
import zipfile
import importlib.metadata

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
import transformers
assert transformers.__version__ == "4.57.1", "Run section 1, restart the Colab session, then rerun from section 2. Loaded Transformers: " + transformers.__version__
from transformers import RobertaTokenizer, T5EncoderModel, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "Select a GPU runtime before continuing. CPU training is disabled."
assert torch.cuda.is_bf16_supported(), "Use an A100 or another GPU that supports bfloat16."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

SEED = 42
MODEL_NAME = "Salesforce/codet5-base"
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 2e-5
PATIENCE = 2
RUN_TRAINING = True  # Run section 8 after reviewing the timing estimate.

WORK = Path.cwd() / "codet5_run"
WORK.mkdir(exist_ok=True)
OUTPUT = WORK / "results"
OUTPUT.mkdir(exist_ok=True)

def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

set_seed()
# A seed helps repeatability; it does not guarantee identical GPU results.


## 3. Load the dataset and saved split
Download the exact dataset and saved split from pinned versions. Verify file hashes, author balance, split counts and separation of assigned groups, exact prompts and code hashes. The classifier receives code, not prompts or labels as features.


In [ ]:
PROJECT_VERSION = "2956a0ad07f49529026cacd33728018d6ebf96ca"
PROJECT_URL = "https://raw.githubusercontent.com/S3eeDTR/LLM-AuthorBench-Assignment/" + PROJECT_VERSION
DATA_URL = "https://raw.githubusercontent.com/LLMauthorbench/LLMauthorbench/6a1c2ac173c774cfbd012f4c81e8d0a51bb61eb7/LLM-AuthorBench.json.zip"

def download(url, filename, expected_hash):
    path = WORK / filename
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual_hash == expected_hash, "File verification failed: " + filename
    return path

archive = download(DATA_URL, "dataset.zip", "e24399b7b05b812c5a1148ef44245348859eaa74b1140523cb695f984337f24c")
split_file = download(PROJECT_URL + "/data/splits.csv", "splits.csv", "8a3c8e2883844295fbddf5b78b96898478f606305972de4feb07a238254df5f4")

with zipfile.ZipFile(archive) as zipped:
    records = json.loads(zipped.read("LLM-AuthorBench.json"))
data = pd.DataFrame(records).rename(columns={"c_code": "code", "model_name": "label"})
data["row_id"] = range(len(data))
labels = sorted(["claude-3.5-haiku", "deepseek-chat", "gemini-2.5-flash-preview-05-20", "gpt-4.1", "llama-3.3-70b-instruct"])
data = data[data["label"].isin(labels)].copy()
data["source_hash"] = data["code"].map(lambda text: hashlib.sha256(text.encode()).hexdigest())
assert len(data) == 20000
assert data.groupby("label").size().eq(4000).all()
assert data["code"].str.strip().ne("").all()
assert not data["source_hash"].duplicated().any()

splits = pd.read_csv(split_file)
data = data.merge(splits, on=["row_id", "label", "source_hash"], validate="one_to_one")
assert len(data) == 20000
assert data.groupby("split").size().to_dict() == {"train": 13150, "validation": 2978, "test": 3872}
for part in ["train", "validation", "test"]:
    assert set(data.loc[data["split"] == part, "label"]) == set(labels)
    for column in ["problem_id", "prompt", "source_hash"]:
        inside = set(data.loc[data["split"] == part, column])
        outside = set(data.loc[data["split"] != part, column])
        assert inside.isdisjoint(outside)
label_to_id = {label: number for number, label in enumerate(labels)}
data["target"] = data["label"].map(label_to_id)
print(pd.crosstab(data["split"], data["label"]))


## 4. Tokenize once
CodeT5 uses its own subword tokenizer, not TF-IDF. Each batch is padded only to its longest example. Tokens after `MAX_LENGTH` are discarded. Keep these settings fixed for a run; changing them creates a different experiment.
We explicitly load RobertaTokenizer, as specified by the CodeT5 checkpoint, to avoid automatic fast-tokenizer conversion.


In [ ]:
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8, return_tensors="pt")
rows_by_split = {}
loaders = {}

for part in ["train", "validation", "test"]:
    rows = data[data["split"] == part].reset_index(drop=True)
    rows_by_split[part] = rows
    tokens = tokenizer(rows["code"].tolist(), truncation=True, max_length=MAX_LENGTH, padding=False)
    examples = []
    for index in range(len(rows)):
        examples.append({
            "input_ids": tokens["input_ids"][index],
            "attention_mask": tokens["attention_mask"][index],
            "labels": int(rows.loc[index, "target"]),
        })
    loaders[part] = DataLoader(
        examples, batch_size=BATCH_SIZE, shuffle=(part == "train"),
        collate_fn=collator, num_workers=0, pin_memory=True,
    )
    print(part, len(rows), "programs;", len(loaders[part]), "batches")


## 5. CodeT5 encoder and classifier
The encoder creates a vector for each token. We average the non-padding vectors, apply dropout and use a linear layer to produce five scores. Cross-entropy trains the encoder and classifier together. The randomly initialized classifier warning is expected; decoder weights are unused.

In [ ]:
class CodeT5Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.encoder.config.d_model, len(labels))

    def forward(self, input_ids, attention_mask):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.classifier(self.dropout(pooled))

set_seed()
model = CodeT5Classifier().cuda()
loss_function = nn.CrossEntropyLoss()
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 6. Evaluation helper — validation during training, test only at the end

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    true_labels = []
    predictions = []
    for batch in loader:
        targets = batch.pop("labels")
        inputs = {key: value.cuda(non_blocking=True) for key, value in batch.items()}
        with torch.autocast("cuda", dtype=torch.bfloat16):
            scores = model(**inputs)
        predictions.extend(scores.argmax(dim=1).cpu().tolist())
        true_labels.extend(targets.tolist())
    metrics = {
        "accuracy": accuracy_score(true_labels, predictions),
        "macro_f1": f1_score(true_labels, predictions, labels=range(5), average="macro", zero_division=0),
    }
    return metrics, true_labels, predictions


## 7. Short GPU timing check (no weight updates)
Five warm-up batches plus twenty timed batches run forward and backward passes. No optimizer step occurs, so this check does not train the model. The estimate excludes validation, downloads, saving and optimizer overhead; it is approximate.

If you run out of GPU memory, restart the runtime and reduce `BATCH_SIZE` to 8. If the estimate is too long, decide on fewer epochs **before** running training. Do not reduce the test set or choose settings after examining test scores.

In [ ]:
model.train()
torch.cuda.reset_peak_memory_stats()
measured_seconds = []
for step, batch in enumerate(loaders["train"]):
    if step == 25:
        break
    targets = batch.pop("labels").cuda()
    inputs = {key: value.cuda() for key, value in batch.items()}
    model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss = loss_function(model(**inputs), targets)
    loss.backward()
    torch.cuda.synchronize()
    if step >= 5:
        measured_seconds.append(time.perf_counter() - started)
model.zero_grad(set_to_none=True)
seconds_per_batch = float(np.mean(measured_seconds))
estimated_minutes = seconds_per_batch * len(loaders["train"]) * EPOCHS / 60
print(f"Measured forward/backward time: {seconds_per_batch:.3f} seconds/batch")
print(f"Approximate training passes for {EPOCHS} epochs: {estimated_minutes:.1f} minutes")
print("Add time for optimizer updates, validation and saving. This is not a guaranteed ETA.")
print("Peak GPU memory (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))


## 8. Train and select the best validation checkpoint
Training is enabled by default. The saved run used five requested epochs and patience two; validation Macro-F1 selected epoch three. No test scores are used for checkpoint selection.

For a fresh repeat, rerun section 5 to recreate the model before this cell. Rerunning only this cell does not reset trained weights. Save your checkpoint before disconnecting; this notebook does not resume an interrupted optimizer.


In [ ]:
# RUN_TRAINING = True
assert RUN_TRAINING, "Read section 7, then set RUN_TRAINING = True to begin."
set_seed()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
best_f1 = -1.0
bad_epochs = 0
history = []
training_started = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    epoch_started = time.perf_counter()
    for batch in tqdm(loaders["train"], desc=f"Epoch {epoch}/{EPOCHS}"):
        targets = batch.pop("labels").cuda(non_blocking=True)
        inputs = {key: value.cuda(non_blocking=True) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = loss_function(model(**inputs), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(targets)
    validation_metrics, _, _ = evaluate(loaders["validation"])
    epoch_record = {
        "epoch": epoch, "training_loss": total_loss / len(rows_by_split["train"]),
        "validation_macro_f1": validation_metrics["macro_f1"],
        "validation_accuracy": validation_metrics["accuracy"],
        "epoch_seconds": time.perf_counter() - epoch_started,
    }
    history.append(epoch_record)
    print(epoch_record)
    pd.DataFrame(history).to_csv(OUTPUT / "training_history.csv", index=False)
    if validation_metrics["macro_f1"] > best_f1:
        best_f1 = validation_metrics["macro_f1"]
        best_epoch = epoch
        bad_epochs = 0
        torch.save(model.state_dict(), OUTPUT / "best_model.pt")
    else:
        bad_epochs += 1
    if bad_epochs >= PATIENCE:
        print("Early stopping: no validation improvement.")
        break
training_seconds = time.perf_counter() - training_started
print("Best epoch:", best_epoch, "Validation Macro-F1:", best_f1)


## 9. Evaluate CodeT5 on the held-out test set


In [ ]:
assert (OUTPUT / "best_model.pt").exists(), "Finish training first."
model.load_state_dict(torch.load(OUTPUT / "best_model.pt", map_location="cpu", weights_only=True))
test_metrics, true_labels, predictions = evaluate(loaders["test"])
print("CodeT5 test scores:", test_metrics)
report = classification_report(true_labels, predictions, labels=range(5), target_names=labels, output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(OUTPUT / "test_class_scores.csv")
pd.DataFrame({
    "row_id": rows_by_split["test"]["row_id"],
    "true": [labels[number] for number in true_labels],
    "predicted": [labels[number] for number in predictions],
}).to_csv(OUTPUT / "test_predictions.csv", index=False)
counts = confusion_matrix(true_labels, predictions, labels=range(5))
pd.DataFrame(counts, index=labels, columns=labels).to_csv(OUTPUT / "test_confusion.csv")

figure, axes = plt.subplots(figsize=(7, 6))
axes.imshow(counts, cmap="Blues")
short_names = ["Claude", "DeepSeek", "Gemini", "GPT-4.1", "Llama"]
axes.set_xticks(range(5), short_names, rotation=35, ha="right")
axes.set_yticks(range(5), short_names)
axes.set_xlabel("Predicted author")
axes.set_ylabel("Actual author")
axes.set_title("CodeT5 test confusion matrix")
for row in range(5):
    for column in range(5):
        color = "white" if counts[row, column] > counts.max() / 2 else "black"
        axes.text(column, row, str(counts[row, column]), ha="center", va="center", color=color)
figure.tight_layout()
figure.savefig(OUTPUT / "test_confusion.png", dpi=180)
plt.show()


## 10. Save and download your results
The small results ZIP excludes the large weights file. Download both if you need the trained model. Training duration includes validation and saving, so do not compare it directly with the classical fit-only CPU timings. This notebook records the GPU, settings, versions and selected epoch.

In [ ]:
settings = {
    "model": MODEL_NAME, "architecture": "encoder + masked mean pooling + dropout + linear head",
    "model_revision": getattr(model.encoder.config, "_commit_hash", None),
    "seed": SEED, "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE,
    "epochs_requested": EPOCHS, "epochs_completed": len(history),
    "learning_rate": LEARNING_RATE, "weight_decay": 0.01,
    "best_epoch": best_epoch, "best_validation_macro_f1": best_f1,
    "training_wall_seconds": training_seconds, "test_metrics": test_metrics,
    "gpu": torch.cuda.get_device_name(0), "project_version": PROJECT_VERSION,
    "labels": labels, "versions": {name: importlib.metadata.version(name) for name in ["torch", "transformers", "numpy", "pandas", "scikit-learn"]},
}
(OUTPUT / "settings.json").write_text(json.dumps(settings, indent=2))
tokenizer.save_pretrained(OUTPUT / "tokenizer")
model.encoder.config.save_pretrained(OUTPUT / "encoder_config")
with zipfile.ZipFile(WORK / "codet5_results.zip", "w", zipfile.ZIP_DEFLATED) as zipped:
    for path in OUTPUT.rglob("*"):
        if path.is_file() and path.name != "best_model.pt":
            zipped.write(path, path.relative_to(OUTPUT))
print("Saved:", WORK / "codet5_results.zip")

# Run in Colab to download the small results bundle.
from google.colab import files
files.download(str(WORK / "codet5_results.zip"))
# Optional, large download:
# files.download(str(OUTPUT / "best_model.pt"))


## Recorded outcome and limits
The completed five-epoch run selected epoch 3 (validation Macro-F1 **0.799756**). Test accuracy was **0.769628** and test Macro-F1 **0.772514** on 3,872 programs. These figures come from the saved results, not a new execution of this notebook. GPU reruns may differ.

This was an exploratory original-code run. No CodeT5 comment-removal experiment was completed. Do not infer how comments affected this model or claim that this is a controlled reproduction of the paper. Its architecture, tokenizer, 256-token limit and training protocol differ.

References: [CodeT5](https://huggingface.co/Salesforce/codet5-base), [dataset and authors' implementation](https://github.com/LLMauthorbench/LLMauthorbench).
